## 4 Подготовка данных к обучению

**Входы**

- Данные после очистки и разбиения
- Спецификации моделей из этапа 3.5
- Таблица выводов и решения по признакам

**Выходы**

- Правила обработки и готовые входы для моделей

**Что делаем**

Заполним пропуски и переведём категории в понятный модели вид. Все нужные для этого значения считаем только по train

In [ ]:
# dev-only: подготовка рабочей папки для интерактивного запуска
from pathlib import Path as _NotebookPath
import os as _notebook_os
_notebook_root = next(
    path for path in (_NotebookPath.cwd(), *_NotebookPath.cwd().parents)
    if (path / 'config/build/project.json').is_file()
)
_notebook_os.chdir(_notebook_root)


#### Как готовим признаки

| Признаки | Что делаем | Зачем |
| --- | --- | --- |
| Числовые | Заполняем пропуски медианой train, добавляем отметки о пропусках и приводим значения к сопоставимому масштабу | Признаки измеряются в разных единицах; редкие высокие значения не удаляем автоматически |
| Номинальные | Заполняем пропуски самым частым значением train, затем создаём отдельный столбец 0/1 для каждой категории | Не задаём порядок номинальным категориям |
| Грейд | Заполняем пропуски самым частым значением train; кодируем A=0, …, G=6 | Сохраняем порядок от A к G |

Кодирование категорий отдельными столбцами 0/1 называют **one-hot**. Самое частое значение — **мода**. Отметка о пропуске — дополнительный столбец, показывающий, что исходное значение было неизвестно; такие отметки создаются для полей с пропусками в train

Для изменения масштаба вычитаем медиану и делим на IQR train. Это преобразование называется **RobustScaler**; оно меньше зависит от крайних значений, чем масштабирование по среднему и стандартному отклонению

Для логистической регрессии переход от A к B и от B к C даёт одинаковую прибавку к сумме вкладов признаков. Это упрощение её формы связи с грейдом. Деревья могут делать разные проверки для разных участков порядка

Все шаги соединяем с моделью в **Pipeline** — цепочку обработки и обучения. Каждый вариант обучает свою цепочку только на train. Это помогает одинаково обрабатывать новые заявки

Ставку по грейду не восстанавливаем, а долю кредита не пересчитываем: происхождение расхождений с формулой неизвестно. `loan_status` в признаки не включаем

In [ ]:
# dev-only: загрузчик Python-кода; в сборке заменяется содержимым файла
exec(compile(__import__('pathlib').Path('src/notebooks/_code/preprocessing.py').read_text(encoding='utf-8'), 'src/notebooks/_code/preprocessing.py', 'exec'), globals())

In [ ]:
from sklearn.base import clone
from threadpoolctl import threadpool_limits

# Ограничиваем число вычислительных потоков для воспроизводимых запусков.
_ml_thread_limit = threadpool_limits(limits=1)
assert set(FEATURES) == set(train.columns) - {'loan_status'}
assert set(NUMERICAL_FEATURES) == set(numerical_columns)
assert set(NOMINAL_FEATURES) == set(nominal_columns) - {'loan_status'}
assert set(ORDINAL_FEATURES) == set(ordinal_columns)
assert train.index.intersection(validation.index).empty
assert train.index.intersection(test.index).empty
assert validation.index.intersection(test.index).empty

X_train = train.loc[:, FEATURES].copy()
y_train = train['loan_status'].astype(int).copy()
X_validation = validation.loc[:, FEATURES].copy()
y_validation = validation['loan_status'].astype(int).copy()
preprocessor_template = make_preprocessor()
audit_preprocessor = clone(preprocessor_template).fit(X_train)
preprocessing_preview = pd.DataFrame(
    audit_preprocessor.transform(X_train.head(5)),
    columns=audit_preprocessor.get_feature_names_out(), index=X_train.head(5).index,
)
split_summary = pd.DataFrame([
    {'Выборка': name, 'Записей': len(frame), 'Доля дефолтов': frame['loan_status'].mean()}
    for name, frame in [('train', train), ('validation', validation), ('test', test)]
]).set_index('Выборка')
display(split_summary.round(4))
display(preprocessing_preview.round(3))